In [1]:
import pandas as pd
import duckdb
import time
import os

In [2]:
os.makedirs("data", exist_ok=True)
os.makedirs("output", exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [3]:
import random
from datetime import datetime, timedelta

random.seed(42)

num_records = 100000

products = [
    " Laptop ",
    "Mouse",
    " Keyboard ",
    " Monitor ",
    " Headphones",
    " USB Cable "
]

categories = [
    " Electronics ",
    "electronics",
    " ELECTRONICS",
    "Accessories ",
    " accessories"
]

store_codes = [
    " ST001 ",
    "OLD002",
    " ST003",
    "OLD004 ",
    "ST005 "
]

start_date = datetime(2026, 1, 1)

data = []

for i in range(num_records):

    transaction_id = i + 1

    store_code = random.choice(store_codes)

    product = random.choice(products)

    category = random.choice(categories)

    quantity = random.randint(1, 10)

    price = round(random.uniform(100, 50000), 2)

    sale_date = start_date + timedelta(
        days=random.randint(0, 180)
    )

    data.append([
        transaction_id,
        store_code,
        product,
        category,
        quantity,
        price,
        sale_date.strftime("%Y-%m-%d")
    ])

raw_df = pd.DataFrame(
    data,
    columns=[
        "transaction_id",
        "store_code",
        "product",
        "category",
        "quantity",
        "price",
        "sale_date"
    ]
)

raw_df.head()

,transaction_id,store_code,product,category,quantity,price,sale_date
0,1,ST001,Laptop,ELECTRONICS,4,11238.22,2026-01-27
1,2,ST005,Laptop,accessories,7,1685.96,2026-01-24
2,3,OLD002,Mouse,accessories,10,1424.14,2026-02-20
3,4,ST005,Monitor,electronics,8,29504.36,2026-01-02
4,5,OLD002,USB Cable,Accessories,6,13965.78,2026-02-25


In [4]:
raw_df.to_csv(
    "data/retail_sales_raw.csv",
    index=False
)

print("Raw dataset saved.")
print("Number of records:", len(raw_df))

Raw dataset saved.
Number of records: 100000


In [5]:
print("Dataset Shape:")
print(raw_df.shape)

print("\nColumn Information:")
print(raw_df.info())

print("\nFirst 10 Records:")
display(raw_df.head(10))

Dataset Shape:
(100000, 7)

Column Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   transaction_id  100000 non-null  int64  
 1   store_code      100000 non-null  object 
 2   product         100000 non-null  object 
 3   category        100000 non-null  object 
 4   quantity        100000 non-null  int64  
 5   price           100000 non-null  float64
 6   sale_date       100000 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 5.3+ MB
None

First 10 Records:


,transaction_id,store_code,product,category,quantity,price,sale_date
0,1,ST001,Laptop,ELECTRONICS,4,11238.22,2026-01-27
1,2,ST005,Laptop,accessories,7,1685.96,2026-01-24
2,3,OLD002,Mouse,accessories,10,1424.14,2026-02-20
3,4,ST005,Monitor,electronics,8,29504.36,2026-01-02
4,5,OLD002,USB Cable,Accessories,6,13965.78,2026-02-25
5,6,ST003,Laptop,Electronics,7,4926.15,2026-03-30
6,7,ST005,Keyboard,Electronics,8,26857.78,2026-04-07
7,8,ST001,Headphones,ELECTRONICS,10,44284.04,2026-04-03
8,9,ST005,Mouse,Electronics,1,33097.04,2026-03-16
9,10,ST001,Mouse,Electronics,7,13970.88,2026-06-12


In [6]:
print("Unique Store Codes:")
print(raw_df["store_code"].unique())

print("\nUnique Categories:")
print(raw_df["category"].unique())

print("\nUnique Products:")
print(raw_df["product"].unique())

Unique Store Codes:
[' ST001 ' 'ST005 ' 'OLD002' ' ST003' 'OLD004 ']

Unique Categories:
[' ELECTRONICS' ' accessories' 'electronics' 'Accessories '
 ' Electronics ']

Unique Products:
[' Laptop ' 'Mouse' ' Monitor ' ' USB Cable ' ' Keyboard ' ' Headphones']


In [7]:
etl_start = time.perf_counter()

etl_df = pd.read_csv(
    "data/retail_sales_raw.csv"
)

print("ETL Extract completed.")
print("Records extracted:", len(etl_df))

ETL Extract completed.
Records extracted: 100000


In [8]:
# Remove leading and trailing spaces
etl_df["store_code"] = etl_df["store_code"].str.strip()
etl_df["product"] = etl_df["product"].str.strip()
etl_df["category"] = etl_df["category"].str.strip()

# Standardize category names
etl_df["category"] = etl_df["category"].str.lower()

# Convert category names to proper format
etl_df["category"] = etl_df["category"].str.title()

# Convert legacy store codes
store_mapping = {
    "OLD002": "ST002",
    "OLD004": "ST004"
}

etl_df["store_code"] = (
    etl_df["store_code"]
    .replace(store_mapping)
)

# Convert numeric columns
etl_df["quantity"] = pd.to_numeric(
    etl_df["quantity"]
)

etl_df["price"] = pd.to_numeric(
    etl_df["price"]
)

# Calculate total sales amount
etl_df["total_amount"] = (
    etl_df["quantity"] *
    etl_df["price"]
)

# Convert date
etl_df["sale_date"] = pd.to_datetime(
    etl_df["sale_date"]
)

print("ETL transformation completed.")

display(etl_df.head())

ETL transformation completed.


,transaction_id,store_code,product,category,quantity,price,sale_date,total_amount
0,1,ST001,Laptop,Electronics,4,11238.22,2026-01-27,44952.88
1,2,ST005,Laptop,Accessories,7,1685.96,2026-01-24,11801.72
2,3,ST002,Mouse,Accessories,10,1424.14,2026-02-20,14241.40
3,4,ST005,Monitor,Electronics,8,29504.36,2026-01-02,236034.88
4,5,ST002,USB Cable,Accessories,6,13965.78,2026-02-25,83794.68


In [9]:
print("Store Codes:")
print(etl_df["store_code"].unique())

print("\nCategories:")
print(etl_df["category"].unique())

print("\nData Types:")
print(etl_df.dtypes)

Store Codes:
['ST001' 'ST005' 'ST002' 'ST003' 'ST004']

Categories:
['Electronics' 'Accessories']

Data Types:
transaction_id             int64
store_code                object
product                   object
category                  object
quantity                   int64
price                    float64
sale_date         datetime64[ns]
total_amount             float64
dtype: object


In [10]:
con = duckdb.connect("output/retail_etl_elt.duckdb")

In [11]:
con.register("etl_dataframe", etl_df)

con.execute("""
CREATE OR REPLACE TABLE etl_sales AS
SELECT *
FROM etl_dataframe
""")

print("ETL data loaded into DuckDB.")

ETL data loaded into DuckDB.


In [12]:
etl_count = con.execute("""
SELECT COUNT(*)
FROM etl_sales
""").fetchone()[0]

print("ETL records:", etl_count)

ETL records: 100000


In [13]:
etl_end = time.perf_counter()

etl_time = etl_end - etl_start

print(f"ETL execution time: {etl_time:.4f} seconds")

ETL execution time: 1.0638 seconds


In [14]:
elt_start = time.perf_counter()

In [15]:
raw_df_elt = pd.read_csv(
    "data/retail_sales_raw.csv"
)

con.register(
    "raw_dataframe",
    raw_df_elt
)

con.execute("""
CREATE OR REPLACE TABLE staging_sales AS
SELECT *
FROM raw_dataframe
""")

print("Raw data loaded into staging table.")

Raw data loaded into staging table.


In [16]:
display(
    con.execute("""
    SELECT *
    FROM staging_sales
    LIMIT 10
    """).df()
)

,transaction_id,store_code,product,category,quantity,price,sale_date
0,1,ST001,Laptop,ELECTRONICS,4,11238.22,2026-01-27
1,2,ST005,Laptop,accessories,7,1685.96,2026-01-24
2,3,OLD002,Mouse,accessories,10,1424.14,2026-02-20
3,4,ST005,Monitor,electronics,8,29504.36,2026-01-02
4,5,OLD002,USB Cable,Accessories,6,13965.78,2026-02-25
5,6,ST003,Laptop,Electronics,7,4926.15,2026-03-30
6,7,ST005,Keyboard,Electronics,8,26857.78,2026-04-07
7,8,ST001,Headphones,ELECTRONICS,10,44284.04,2026-04-03
8,9,ST005,Mouse,Electronics,1,33097.04,2026-03-16
9,10,ST001,Mouse,Electronics,7,13970.88,2026-06-12


In [17]:
con.execute("""
CREATE OR REPLACE TABLE elt_sales AS

SELECT
    transaction_id,

    CASE
        WHEN TRIM(store_code) = 'OLD002'
            THEN 'ST002'
        WHEN TRIM(store_code) = 'OLD004'
            THEN 'ST004'
        ELSE TRIM(store_code)
    END AS store_code,

    TRIM(product) AS product,

    CASE
        WHEN LOWER(TRIM(category)) = 'electronics'
            THEN 'Electronics'
        WHEN LOWER(TRIM(category)) = 'accessories'
            THEN 'Accessories'
        ELSE TRIM(category)
    END AS category,

    CAST(quantity AS INTEGER) AS quantity,

    CAST(price AS DOUBLE) AS price,

    CAST(quantity AS INTEGER)
        * CAST(price AS DOUBLE)
        AS total_amount,

    CAST(sale_date AS DATE) AS sale_date

FROM staging_sales
""")

print("ELT transformation completed using DuckDB.")

ELT transformation completed using DuckDB.


In [18]:
display(
    con.execute("""
    SELECT *
    FROM elt_sales
    LIMIT 10
    """).df()
)

,transaction_id,store_code,product,category,quantity,price,total_amount,sale_date
0,1,ST001,Laptop,Electronics,4,11238.22,44952.88,2026-01-27
1,2,ST005,Laptop,Accessories,7,1685.96,11801.72,2026-01-24
2,3,ST002,Mouse,Accessories,10,1424.14,14241.40,2026-02-20
3,4,ST005,Monitor,Electronics,8,29504.36,236034.88,2026-01-02
4,5,ST002,USB Cable,Accessories,6,13965.78,83794.68,2026-02-25
5,6,ST003,Laptop,Electronics,7,4926.15,34483.05,2026-03-30
6,7,ST005,Keyboard,Electronics,8,26857.78,214862.24,2026-04-07
7,8,ST001,Headphones,Electronics,10,44284.04,442840.40,2026-04-03
8,9,ST005,Mouse,Electronics,1,33097.04,33097.04,2026-03-16
9,10,ST001,Mouse,Electronics,7,13970.88,97796.16,2026-06-12


In [19]:
print("ELT Store Codes:")

print(
    con.execute("""
    SELECT DISTINCT store_code
    FROM elt_sales
    ORDER BY store_code
    """).df()
)

print("\nELT Categories:")

print(
    con.execute("""
    SELECT DISTINCT category
    FROM elt_sales
    ORDER BY category
    """).df()
)

ELT Store Codes:
  store_code
0      ST001
1      ST002
2      ST003
3      ST004
4      ST005

ELT Categories:
      category
0  Accessories
1  Electronics


In [20]:
elt_end = time.perf_counter()

elt_time = elt_end - elt_start

print(f"ELT execution time: {elt_time:.4f} seconds")

ELT execution time: 1.5770 seconds


In [21]:
etl_count = con.execute("""
SELECT COUNT(*)
FROM etl_sales
""").fetchone()[0]

elt_count = con.execute("""
SELECT COUNT(*)
FROM elt_sales
""").fetchone()[0]

print("ETL records:", etl_count)
print("ELT records:", elt_count)

ETL records: 100000
ELT records: 100000


In [22]:
etl_result = con.execute("""
SELECT *
FROM etl_sales
ORDER BY transaction_id
""").df()

elt_result = con.execute("""
SELECT *
FROM elt_sales
ORDER BY transaction_id
""").df()

print("Same number of records:",
      len(etl_result) == len(elt_result))

print("Same columns:",
      list(etl_result.columns) == list(elt_result.columns))

Same number of records: True
Same columns: False


In [23]:
print("ETL data types:")
print(etl_result.dtypes)

print("\nELT data types:")
print(elt_result.dtypes)

ETL data types:
transaction_id             int64
store_code                object
product                   object
category                  object
quantity                   int64
price                    float64
sale_date         datetime64[ns]
total_amount             float64
dtype: object

ELT data types:
transaction_id             int64
store_code                object
product                   object
category                  object
quantity                   int32
price                    float64
total_amount             float64
sale_date         datetime64[us]
dtype: object


In [24]:
for column in etl_result.columns:
    same = etl_result[column].astype(str).equals(
        elt_result[column].astype(str)
    )
    
    print(column, ":", same)

transaction_id : True
store_code : True
product : True
category : True
quantity : True
price : True
sale_date : True
total_amount : True


In [25]:
# Make copies
etl_final = etl_result.copy()
elt_final = elt_result.copy()

# Use the same column order for both DataFrames
common_columns = sorted(
    set(etl_final.columns) & set(elt_final.columns)
)

etl_final = etl_final[common_columns]
elt_final = elt_final[common_columns]

# Reset row indexes
etl_final = etl_final.reset_index(drop=True)
elt_final = elt_final.reset_index(drop=True)

# Convert values to strings for logical comparison
etl_final = etl_final.astype(str)
elt_final = elt_final.astype(str)

print("ETL columns:")
print(list(etl_final.columns))

print("\nELT columns:")
print(list(elt_final.columns))

print("\nSame number of records:",
      len(etl_final) == len(elt_final))

print("Same columns:",
      list(etl_final.columns) == list(elt_final.columns))

print("ETL and ELT data values identical:",
      etl_final.equals(elt_final))

ETL columns:
['category', 'price', 'product', 'quantity', 'sale_date', 'store_code', 'total_amount', 'transaction_id']

ELT columns:
['category', 'price', 'product', 'quantity', 'sale_date', 'store_code', 'total_amount', 'transaction_id']

Same number of records: True
Same columns: True
ETL and ELT data values identical: True


In [26]:
performance = pd.DataFrame({
    "Pipeline": ["ETL", "ELT"],
    "Execution Time (seconds)": [
        etl_time,
        elt_time
    ],
    "Records Processed": [
        etl_count,
        elt_count
    ]
})

display(performance)

,Pipeline,Execution Time (seconds),Records Processed
0,ETL,1.063848,100000
1,ELT,1.576983,100000


In [27]:
difference = abs(etl_time - elt_time)

print(f"ETL execution time: {etl_time:.4f} seconds")
print(f"ELT execution time: {elt_time:.4f} seconds")
print(f"Difference: {difference:.4f} seconds")

ETL execution time: 1.0638 seconds
ELT execution time: 1.5770 seconds
Difference: 0.5131 seconds


In [28]:
if etl_time < elt_time:
    print("ETL was faster in this local benchmark.")
elif elt_time < etl_time:
    print("ELT was faster in this local benchmark.")
else:
    print("Both pipelines had the same execution time.")

ETL was faster in this local benchmark.


In [29]:
etl_throughput = etl_count / etl_time
elt_throughput = elt_count / elt_time

print(f"ETL throughput: {etl_throughput:.2f} records/second")
print(f"ELT throughput: {elt_throughput:.2f} records/second")


ETL throughput: 93998.43 records/second
ELT throughput: 63412.24 records/second


In [31]:
print("Staging table record count:")

print(
    con.execute("""
    SELECT COUNT(*) AS record_count
    FROM staging_sales
    """).df()
)

Staging table record count:
   record_count
0        100000


In [32]:
display(
    con.execute("""
    SELECT *
    FROM staging_sales
    LIMIT 10
    """).df()
)

,transaction_id,store_code,product,category,quantity,price,sale_date
0,1,ST001,Laptop,ELECTRONICS,4,11238.22,2026-01-27
1,2,ST005,Laptop,accessories,7,1685.96,2026-01-24
2,3,OLD002,Mouse,accessories,10,1424.14,2026-02-20
3,4,ST005,Monitor,electronics,8,29504.36,2026-01-02
4,5,OLD002,USB Cable,Accessories,6,13965.78,2026-02-25
5,6,ST003,Laptop,Electronics,7,4926.15,2026-03-30
6,7,ST005,Keyboard,Electronics,8,26857.78,2026-04-07
7,8,ST001,Headphones,ELECTRONICS,10,44284.04,2026-04-03
8,9,ST005,Mouse,Electronics,1,33097.04,2026-03-16
9,10,ST001,Mouse,Electronics,7,13970.88,2026-06-12


In [33]:
display(
    con.execute("""
    SELECT *
    FROM elt_sales
    LIMIT 10
    """).df()
)

,transaction_id,store_code,product,category,quantity,price,total_amount,sale_date
0,1,ST001,Laptop,Electronics,4,11238.22,44952.88,2026-01-27
1,2,ST005,Laptop,Accessories,7,1685.96,11801.72,2026-01-24
2,3,ST002,Mouse,Accessories,10,1424.14,14241.40,2026-02-20
3,4,ST005,Monitor,Electronics,8,29504.36,236034.88,2026-01-02
4,5,ST002,USB Cable,Accessories,6,13965.78,83794.68,2026-02-25
5,6,ST003,Laptop,Electronics,7,4926.15,34483.05,2026-03-30
6,7,ST005,Keyboard,Electronics,8,26857.78,214862.24,2026-04-07
7,8,ST001,Headphones,Electronics,10,44284.04,442840.40,2026-04-03
8,9,ST005,Mouse,Electronics,1,33097.04,33097.04,2026-03-16
9,10,ST001,Mouse,Electronics,7,13970.88,97796.16,2026-06-12


In [34]:
con.execute("""
CREATE OR REPLACE TABLE elt_sales_buggy AS

SELECT
    transaction_id,

    CASE
        WHEN TRIM(store_code) = 'OLD002'
            THEN 'ST999'
        WHEN TRIM(store_code) = 'OLD004'
            THEN 'ST004'
        ELSE TRIM(store_code)
    END AS store_code,

    TRIM(product) AS product,

    CASE
        WHEN LOWER(TRIM(category)) = 'electronics'
            THEN 'Electronics'
        WHEN LOWER(TRIM(category)) = 'accessories'
            THEN 'Accessories'
        ELSE TRIM(category)
    END AS category,

    CAST(quantity AS INTEGER) AS quantity,
    CAST(price AS DOUBLE) AS price,

    CAST(quantity AS INTEGER)
        * CAST(price AS DOUBLE)
        AS total_amount,

    CAST(sale_date AS DATE) AS sale_date

FROM staging_sales
""")

print("Buggy ELT transformation created.")

Buggy ELT transformation created.


In [35]:
display(
    con.execute("""
    SELECT DISTINCT store_code
    FROM elt_sales_buggy
    ORDER BY store_code
    """).df()
)

,store_code
0,ST001
1,ST003
2,ST004
3,ST005
4,ST999


In [36]:
con.execute("""
CREATE OR REPLACE TABLE elt_sales_fixed AS

SELECT
    transaction_id,

    CASE
        WHEN TRIM(store_code) = 'OLD002'
            THEN 'ST002'
        WHEN TRIM(store_code) = 'OLD004'
            THEN 'ST004'
        ELSE TRIM(store_code)
    END AS store_code,

    TRIM(product) AS product,

    CASE
        WHEN LOWER(TRIM(category)) = 'electronics'
            THEN 'Electronics'
        WHEN LOWER(TRIM(category)) = 'accessories'
            THEN 'Accessories'
        ELSE TRIM(category)
    END AS category,

    CAST(quantity AS INTEGER) AS quantity,
    CAST(price AS DOUBLE) AS price,

    CAST(quantity AS INTEGER)
        * CAST(price AS DOUBLE)
        AS total_amount,

    CAST(sale_date AS DATE) AS sale_date

FROM staging_sales
""")

print("Transformation fixed successfully.")

Transformation fixed successfully.


In [37]:
display(
    con.execute("""
    SELECT DISTINCT store_code
    FROM elt_sales_fixed
    ORDER BY store_code
    """).df()
)

,store_code
0,ST001
1,ST002
2,ST003
3,ST004
4,ST005
